# DNN - HW 1 - Jakub Pniewski - 459481

In [ ]:
! wget https://github.com/marcin119a/data/raw/refs/heads/main/data_gsn.zip
! unzip data_gsn.zip &> /dev/null
! rm data_gsn.zip

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.io as io
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
from copy import deepcopy


import pandas as pd
import numpy as np

In [ ]:
FIGSIZE = (12, 10)
PX_SIZE = (1280, 720)

N_PAIR_CLASSES = 15
N_CLS_CLASSES = 135
N_CNT_CLASSES = 6
TRAIN_SIZE = 9000
DATASET_SIZE = 10000

N_EPOCHS = 100
TRAIN_BATCH_SIZE = 64
VAL_BATCH_SIZE = 1000

torch.manual_seed(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32

### Ploting functions

boring stuff

In [ ]:
def show_image(
    image,
    title=None,
    figsize=FIGSIZE
):
    plt.figure(figsize=figsize)
    plt.imshow(image.permute(1, 2, 0))
    plt.axis('off')
    if title is not None:
        plt.title(title)
    plt.show()


def get_metrics_fig(
    metrics_dict,
    title="Metrics"
):
    df = pd.DataFrame(metrics_dict)
    df['Epoch'] = df.index

    df_long = df.melt(id_vars='Epoch', var_name='Metric', value_name='Value')

    fig = px.line(
        df_long,
        x='Epoch',
        y='Value',
        color='Metric',
        title=title,
        labels={'Value': 'Metric Value'}
    )
    return fig


def get_metric_fig(
    metric_values,
    title,
):
    fig = px.line(
        y=metric_values,
        title=title,
        labels={'x': 'Epoch', 'y': title}
    )
    return fig


def get_bar_fig(
    metric_values,
    title,
    xlabel,
    ylabel,
):
    x = np.arange(len(metric_values))
    y = metric_values
    fig = px.bar(
        x=x,
        y=y,
        title=title,
        labels={'x': xlabel, 'y': ylabel}
    )
    return fig


def plot_figs(
    figs,
    grid_shape,
    title="Metrics",
    px_size=PX_SIZE
):
    fig = make_subplots(
        rows=grid_shape[0],
        cols=grid_shape[1],
        subplot_titles=[f.layout.title.text for f in figs]
    )
    for r in range(grid_shape[0]):
        for c in range(grid_shape[1]):
            if r * grid_shape[1] + c >= len(figs):
                break
            sub_fig = figs[r * grid_shape[1] + c]
            for trace in sub_fig.data:
                fig.add_trace(
                    trace,
                    row=r + 1,
                    col=c + 1
                )
    fig.update_layout(title_text=title)
    fig.update_layout(width=px_size[0], height=px_size[1])
    fig.show()


def plot_metric(
    metric_values,
    title,
    px_size=PX_SIZE
):
    fig = get_metric_fig(
        metric_values,
        title
    )
    fig.update_layout(width=px_size[0], height=px_size[1])
    fig.show()


def plot_bar(
    metric_values,
    title,
    xlabel,
    ylabel,
    px_size=PX_SIZE
):
    fig = get_bar_fig(
        metric_values,
        title,
        xlabel,
        ylabel
    )
    fig.update_layout(width=px_size[0], height=px_size[1])
    fig.show()


def confusion_matrix_fig(cm, n_classes, title="Confusion Matrix"):
    fig = px.imshow(
        cm,
        labels=dict(x="Predicted Label", y="True Label", color="Count"),
        x=np.arange(n_classes),
        y=np.arange(n_classes),
        title=title
    )
    return fig

### Map mask to class

I have to be able to map the original amounts (6 classes, exactly two values are greater than 0 and sum to 10) into 135 classes.

This is a pretty simple method:
- first I encode the binary classes in "lexicographical" order (for example (0 0 4 0 6 0) is encoded as (0 0 1 0 1 0) and is before (0 0 0 1 1 0) but after (0 0 1 1 0 0)). I can do it with formula $f(i_0, i_2) = i_0 \cdot (9 - i_0) / 2 + i_1 - 1$, where $i_0$ is the index of the first non zero element and $i_1$ is the index of the second non zero element.
- then I multiply the first non zero element by 9 and add the value at the second non zero element minus 1 (then it's in range 0-8).

This way i can also easily tell which pair it was - by deviding by 9 with integer division.

I also implemented some tests to be sure I don't hava an off by one.

In [ ]:
def amounts_to_class(amounts):
    i0, i1 = torch.nonzero(amounts)
    cnt_0 = amounts[i0]

    mask_encoding = i0 * (9 - i0) / 2 + i1 - 1
    return (9 * mask_encoding + cnt_0 - 1).to(torch.long)


def class_to_pair_encoding(cls):
    return cls // 9


class_to_shapes = {}

s = set()
for i in range(5):
    for j in range(i + 1, 6):
        for k in range(1, 10):
            # k times elemnt i and 10 - k times element j
            t = torch.zeros(6)
            t[i] = k
            t[j] = 10 - k
            cls = amounts_to_class(t)

            s.add(cls)
            class_to_shapes[cls.item()] = (i, j)

assert len(s) == 135 and min(s) == 0 and max(s) == 134

s_a = set(range(135))
s = set(i.item() for i in s)

assert s == s_a

### Model definition

In [ ]:
class GeGLU(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.layernorm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(dim, 2 * dim)
        self.gelu = nn.GELU()

    def forward(self, x):
        h = self.layernorm(x)
        h, gate = self.linear(h).chunk(2, dim=-1)
        h = self.dropout(h)
        h *= self.gelu(gate)
        return x + h


class Model(nn.Module):
    def __init__(
        self,
        n_cls_classes=N_CLS_CLASSES,
        n_cnt_classes=N_CNT_CLASSES,
        dropout=0.4
    ):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=1, padding=1), nn.ReLU(),
            nn.Flatten(start_dim=1),
            nn.Linear(64 * 28 * 28, 256), nn.ReLU()
        )

        self.n_cls_classes = n_cls_classes
        self.n_cnt_classes = n_cnt_classes

        self.head_cls = nn.Sequential(
            GeGLU(256, dropout),
            GeGLU(256, dropout),
            nn.Dropout(dropout),
            nn.Linear(256, n_cls_classes),
            nn.LogSoftmax(dim=1)
        )
        self.head_cnt = nn.Sequential(
            GeGLU(256, dropout),
            GeGLU(256, dropout),
            nn.Dropout(dropout),
            nn.Linear(256, n_cnt_classes)
        )

    def forward(self, x):
        x = x / 128.0 - 1.0
        x = self.backbone(x)
        out_cls = self.head_cls(x)
        out_cnt = self.head_cnt(x)
        return out_cls, out_cnt

    def forward_cls(self, x):
        x = x / 128.0 - 1.0
        x = self.backbone(x)
        out_cls = self.head_cls(x)
        return (
            out_cls,
            torch.zeros(
                (x.shape[0], self.n_cnt_classes),
                device=x.device,
                dtype=x.dtype
            )
        )

    def forward_cnt(self, x):
        x = x / 128.0 - 1.0
        x = self.backbone(x)
        out_cnt = self.head_cnt(x)
        return torch.zeros(
            (x.shape[0], self.n_cls_classes),
            device=x.device,
            dtype=x.dtype
        ), out_cnt


model = Model()

### Data augmentation

I have implemented 3 data augmentations:
- horizontal flip with probability 0.5
- vertical flip with probability 0.5
- rotation by 90 degrees with probability 2/3

Rotation has probability 2/3 because it can be done in 2 ways (clockwise and counterclockwise) and I decide which way with equal probability. So all possibilities (no rotation, clockwise, counterclockwise) have equal probability of 1/3.

In [ ]:
class Augmentor(nn.Module):
    def __init__(self, p):
        super().__init__()
        self.p = p

    def augment(self, x, y):
        raise NotImplementedError()

    def forward(self, x, y):
        if torch.rand(1).item() >= self.p:
            return x, y
        return self.augment(x, y.clone())


class HorizontalFlipAugmentor(Augmentor):
    def augment(self, x, y):
        x = torch.flip(x, dims=[2])
        # swap triangle right/left
        y[[3, 5]] = y[[5, 3]]
        return x, y


class VerticalFlipAugmentor(Augmentor):
    def augment(self, x, y):
        x = torch.flip(x, dims=[1])
        # swap triangle up/down
        y[[2, 4]] = y[[4, 2]]
        return x, y


class Rotation90Augmentor(Augmentor):
    def augment(self, x, y):
        side = torch.randint(0, 2, (1,)).item()
        if side == 0:  # clockwise
            x = torch.rot90(x, -1, (1, 2))
            y[[2, 3, 4, 5]] = y[[5, 2, 3, 4]]
        else:  # counterclockwise
            x = torch.rot90(x, 1, (1, 2))
            y[[2, 3, 4, 5]] = y[[3, 4, 5, 2]]
        return x, y

### Dataset

Because the dataset is quite small and the pictures have a low resolution, I decided to load the entire dataset into gpu vram at once. It easily fits and is much faster than loading on the fly. Not too professional, but works well for a small project like this.

Inside the Dataset class, I include the augmentations. I switch them off for validation.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, r, augment, device, dtype):
        data = pd.read_csv('data/labels.csv')
        self.data = data.iloc[r]
        self.images = [
            io.read_image(
                'data/' + name,
                mode=io.ImageReadMode.GRAY
            ).to(device, dtype) for name in self.data['name']
        ]
        self.amounts = [
            torch.from_numpy(
                row.drop('name').values.astype(np.float32)
            ).to(device, dtype)
            for _, row in self.data.iterrows()
        ]
        self.augment = augment
        self.augmentations = [
            HorizontalFlipAugmentor(0.5),
            VerticalFlipAugmentor(0.5),
            Rotation90Augmentor(2/3),
        ]
        self.device = device
        self.dtype = dtype

    def __getitem__(self, idx):
        img = self.images[idx]
        amounts = self.amounts[idx]

        if self.augment:
            for augmentor in self.augmentations:
                img, amounts = augmentor(img, amounts)

        cls = amounts_to_class(amounts)
        return (img, cls, amounts)

    def __len__(self):
        return len(self.data)

    @staticmethod
    def collate_fn(batch):
        imgs, masks, amounts = zip(*batch)
        imgs = torch.stack(imgs)
        cls = torch.cat(masks)
        amounts = torch.stack(amounts)
        return imgs, cls, amounts

In [ ]:
train_data = ImageDataset(
    range(TRAIN_SIZE),
    augment=True,
    device=device,
    dtype=dtype
)
val_data = ImageDataset(
    range(TRAIN_SIZE, DATASET_SIZE),
    augment=False,
    device=device,
    dtype=dtype
)

### Check augmentation correctness

It's good to visually check if the augmentations are correct.

In [ ]:
horizontal = HorizontalFlipAugmentor(1)
vertical = VerticalFlipAugmentor(1)
rotation90 = Rotation90Augmentor(1)

image, _, amounts = val_data[2]

image = image.to('cpu')
amounts = amounts.to('cpu')

h_img, h_amounts = horizontal(image, amounts)
v_img, v_amounts = vertical(image, amounts)
r1_img, r1_amounts = rotation90(image, amounts)
r2_img, r2_amounts = rotation90(image, amounts)
while torch.equal(r2_img, r1_img):
    r2_img, r2_amounts = rotation90(image, amounts)

show_image(image, "Original image", (4, 4))
print("Original label:", amounts)
show_image(h_img, "Horizontal flip", (4, 4))
print("Horizontal flip label:", h_amounts)
show_image(v_img, "Vertical flip", (4, 4))
print("Vertical flip label:", v_amounts)
show_image(r1_img, "Rotation 1", (4, 4))
print("Rotation 1 flip label:", r1_amounts)
show_image(r2_img, "Rotation 2", (4, 4))
print("Rotation 2 flip label:", r2_amounts)

### Exploratory Data Analysis

first I we should check the distribution of classes in the dataset.

In [ ]:
how_many = torch.zeros(N_CLS_CLASSES, dtype=torch.int32)

train_data.augment = False

for _, cls, _ in train_data:
    how_many[cls.to("cpu")] += 1

train_data.augment = True


plot_bar(
    how_many.numpy(),
    title="Number of samples per class in training set",
    xlabel="Class",
    ylabel="Number of samples"
)

We can see that the distribution is pretty balanced. However, some pairs do not occur at all. Augmentations I apply will make the dataset even more balanced as every triangle direction can be transformed into any other.

We should also check if the distribution in the regression task is balanced. To that I will count how many times each number from 1 to 9 is present in the dataset.

In [ ]:
occurences = torch.zeros(10, dtype=torch.float32)

train_data.augment = False

for _, cls, amounts in train_data:
    amounts = amounts.to('cpu')
    i0, i1 = torch.nonzero(amounts)
    occurences[int(amounts[i0])] += 1
    occurences[int(amounts[i1])] += 1

train_data.augment = True

plot_bar(
    occurences.numpy(),
    title=f"Distribution of counts in training set",
    xlabel="Count",
    ylabel="Number of occurences"
)

We can see that the number of occurences form a uniform distribution, which is good. This also explains the "gaps" in the classification task - in the datase there is never a split (1, 9) of the 10 geometric shapes, so classes corresponding to that split do not occur at all.

I've also previously checked that this is true for all 6 classes. But I don't want this notebook to be too long.

### Metrics

I implemented all metrics that were specified in the task description. I decided to implement them as classes which keep track of the values. Each metric class also has a method to plot it.

In [ ]:
from abc import ABC, abstractmethod


class Metric(ABC):
    def __init__(self):
        self.values = []
    name: str
    values: list

    @abstractmethod
    def __call__(self, preds, targets):
        pass

    @abstractmethod
    def plot(self, metric_values):
        pass


class Top1Accuracy(Metric):
    name: str = "Top-1 accuracy"

    def __call__(self, preds, targets):
        preds = preds[0]  # classification
        targets = targets[0]
        argmax = torch.argmax(preds, dim=-1)
        correct = argmax == targets
        accuracy = torch.sum(correct) / targets.numel()
        self.values.append(accuracy.item())
        return accuracy.item()

    def plot(self, metric_values):
        return get_metric_fig(
            metric_values,
            title=self.name
        )


class PerPairAccuracy(Metric):
    name: str = "Per-pair accuracy"

    def __call__(self, preds, targets):
        preds = preds[0]  # classification
        targets = targets[0]
        argmax = torch.argmax(preds, dim=-1)
        correct = argmax == targets

        pair_classes = class_to_pair_encoding(targets)
        class_counts = pair_classes.bincount(minlength=N_PAIR_CLASSES)
        correct_counts = pair_classes.bincount(
            correct, minlength=N_PAIR_CLASSES)

        accuracies = correct_counts / class_counts
        self.values = accuracies.cpu().tolist()
        return accuracies.cpu().tolist()

    def plot(self, metric_values):
        return get_bar_fig(
            metric_values,
            title=self.name,
            xlabel="Pair class",
            ylabel="Accuracy"
        )


class MacroF1Score(Metric):
    name: str = "Macro F1 Score"

    def __call__(self, preds, targets):
        preds = preds[0]  # classification
        targets = targets[0]
        argmax = torch.argmax(preds, dim=-1)
        correct = argmax == targets
        incorrect = argmax != targets

        TP = targets.bincount(correct, minlength=N_CLS_CLASSES)
        FP = argmax.bincount(minlength=N_CLS_CLASSES) - TP
        FN = targets.bincount(incorrect, minlength=N_CLS_CLASSES)

        eps = 1e-9
        precision = TP / (TP + FP + eps)
        recall = TP / (TP + FN + eps)

        f1 = 2 * precision * recall / (precision + recall + eps)
        macro_f1 = torch.mean(f1)
        self.values.append(macro_f1.item())
        return macro_f1

    def plot(self, metric_values):
        return get_metric_fig(
            metric_values,
            title=self.name
        )


class RMSEPerClass(Metric):
    name: str = "RMSE per class"

    def __call__(self, preds, targets):
        preds = preds[1]  # counts
        targets = targets[1]
        loss = F.mse_loss(preds, targets, reduction="none")
        mse = torch.mean(loss, dim=0)  # mean along batch
        self.values = torch.sqrt(mse).cpu().tolist()
        return self.values

    def plot(self, metric_values):
        return get_bar_fig(
            metric_values,
            title=self.name,
            xlabel="Count class",
            ylabel="RMSE"
        )


class RMSE(Metric):
    name: str = "RMSE"

    def __call__(self, preds, targets):
        preds = preds[1]  # counts
        targets = targets[1]
        loss = F.mse_loss(preds, targets, reduction='none')
        mse = torch.mean(loss)
        rmse = torch.sqrt(mse).item()
        self.values.append(rmse)
        return rmse

    def plot(self, metric_values):
        return get_metric_fig(
            metric_values,
            title=self.name
        )


class MAEPerClass(Metric):
    name: str = "MAE per class"

    def __call__(self, preds, targets):
        preds = preds[1]  # counts
        targets = targets[1]
        mae = F.l1_loss(preds, targets, reduction='none')
        self.values = torch.mean(mae, dim=0).cpu().tolist()
        return self.values

    def plot(self, metric_values):
        return get_bar_fig(
            metric_values,
            title=self.name,
            xlabel="Count class",
            ylabel="MAE"
        )


class MAE(Metric):
    name: str = "MAE"

    def __call__(self, preds, targets):
        preds = preds[1]  # counts
        targets = targets[1]
        mae = F.l1_loss(preds, targets).item()
        self.values.append(mae)
        return mae

    def plot(self, metric_values):
        return get_metric_fig(
            metric_values,
            title=self.name
        )

In [ ]:
# Classification
def top_1_accuracy(preds, targets):
    argmax = torch.argmax(preds, dim=-1)
    correct = argmax == targets
    accuracy = torch.sum(correct) / targets.numel()
    return accuracy.item()


def per_pair_accuracy(preds, targets):
    argmax = torch.argmax(preds, dim=-1)
    correct = argmax == targets

    pair_classes = class_to_pair_encoding(targets)
    class_counts = pair_classes.bincount(minlength=N_PAIR_CLASSES)
    correct_counts = pair_classes.bincount(correct, minlength=N_PAIR_CLASSES)

    accuracies = correct_counts / class_counts
    return accuracies


def macro_f1_score(preds, targets):
    argmax = torch.argmax(preds, dim=-1)
    correct = argmax == targets
    incorrect = argmax != targets

    TP = targets.bincount(correct, minlength=N_CLS_CLASSES)
    FP = argmax.bincount(minlength=N_CLS_CLASSES) - TP
    FN = targets.bincount(incorrect, minlength=N_CLS_CLASSES)

    eps = 1e-9
    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)

    f1 = 2 * precision * recall / (precision + recall + eps)
    macro_f1 = torch.mean(f1)

    return macro_f1

# Regression


def rmse_per_class(preds, targets):
    loss = F.mse_loss(preds, targets, reduction="none")
    mse = torch.mean(loss, dim=0)  # mean along batch
    return torch.sqrt(mse)


def rmse(preds, targets):
    mse = F.mse_loss(preds, targets)
    return torch.sqrt(mse)


def mae_per_class(preds, targets):
    loss = F.l1_loss(preds, targets, reduction="none")
    mae = torch.mean(loss, dim=0)  # mean along batch
    return mae


def mae(preds, targets):
    return F.l1_loss(preds, targets).item()

### Trainer

I implemented a trainer class instead of just a function or loop. This way I can easily launch multiple trainings with different parameters to compare them. It's also just a habit.

I aslo implemented loss functions. The loss for multitask learning can be easily used to train classification only model by setting lambda to 0. I had to implement a separate class for regression loss. It looks a bit weird but it enables me to use the same trainer.

In [ ]:
import time


class Trainer:
    def __init__(
        self,
        model: nn.Module,
        train_dataloader: DataLoader,
        eval_dataloader: DataLoader,
        loss_fn: callable,
        eval_metrics: list,
        optimizer: torch.optim.Optimizer,
        n_epochs: int,
        patience: int
    ):
        self.model = model
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader
        self.loss_fn = loss_fn
        self.eval_metrics = eval_metrics
        self.optimizer = optimizer
        self.n_epochs = n_epochs
        self.patience = patience

    def print_losses(
        self,
        avg_whole_loss,
        avg_cls_loss,
        avg_cnt_loss,
    ):
        print(f"    Loss: {avg_whole_loss:.4f}")
        print(f"    Classification : {avg_cls_loss:.4f}")
        print(f"    Regression: {avg_cnt_loss:.4f}")

    @torch.no_grad()
    def validation(self):
        all_preds = []
        all_targets = []

        avg_whole_loss = 0.0
        avg_cls_loss = 0.0
        avg_cnt_loss = 0.0

        self.model.eval()
        for batch in self.eval_dataloader:
            x, y_cls, y_cnt = batch
            pred_cls, pred_cnt = self.model(x)

            all_preds.append((pred_cls, pred_cnt))
            all_targets.append((y_cls, y_cnt))

            whole_loss, cls_loss, cnt_loss = self.loss_fn(
                (pred_cls, pred_cnt), (y_cls, y_cnt))

            avg_whole_loss += whole_loss.item() * x.shape[0]
            avg_cls_loss += cls_loss.item() * x.shape[0]
            avg_cnt_loss += cnt_loss.item() * x.shape[0]

        avg_whole_loss /= len(self.eval_dataloader.dataset)
        avg_cls_loss /= len(self.eval_dataloader.dataset)
        avg_cnt_loss /= len(self.eval_dataloader.dataset)

        all_preds = (
            torch.cat([p[0] for p in all_preds]),
            torch.cat([p[1] for p in all_preds])
        )
        all_targets = (
            torch.cat([t[0] for t in all_targets]),
            torch.cat([t[1] for t in all_targets])
        )
        print("Validation:")
        self.print_losses(
            avg_whole_loss,
            avg_cls_loss,
            avg_cnt_loss
        )

        for metric in self.eval_metrics:
            result = metric(all_preds, all_targets)
            self.eval_metrics_values[metric.name].append(result)
            print(f"    {metric.name}: {result}")

        self.eval_losses.append(avg_whole_loss)
        self.eval_cls_losses.append(avg_cls_loss)
        self.eval_cnt_losses.append(avg_cnt_loss)
        return avg_whole_loss

    def train(self):
        start_time = time.time()

        best_eval_loss = float('inf')
        best_eval_loss_epoch = -1

        self.train_losses = []
        self.train_cls_losses = []
        self.train_cnt_losses = []

        self.eval_losses = []
        self.eval_cls_losses = []
        self.eval_cnt_losses = []

        self.eval_metrics_values = {
            metric.name: [] for metric in self.eval_metrics
        }

        for epoch in range(self.n_epochs):
            print(f'Epoch {epoch+1}/{self.n_epochs}')
            avg_whole_loss = 0.0
            avg_cls_loss = 0.0
            avg_cnt_loss = 0.0

            self.model.train()

            for batch in self.train_dataloader:
                self.optimizer.zero_grad()

                x, y_cls, y_cnt = batch
                pred_cls, pred_cnt = self.model(x)

                whole_loss, cls_loss, cnt_loss = self.loss_fn(
                    (pred_cls, pred_cnt), (y_cls, y_cnt))

                avg_whole_loss += whole_loss.item() * x.shape[0]
                avg_cls_loss += cls_loss.item() * x.shape[0]
                avg_cnt_loss += cnt_loss.item() * x.shape[0]

                whole_loss.backward()
                self.optimizer.step()

            avg_whole_loss /= len(self.train_dataloader.dataset)
            avg_cls_loss /= len(self.train_dataloader.dataset)
            avg_cnt_loss /= len(self.train_dataloader.dataset)

            self.train_losses.append(avg_whole_loss)
            self.train_cls_losses.append(avg_cls_loss)
            self.train_cnt_losses.append(avg_cnt_loss)

            print("Train loss:")
            self.print_losses(
                avg_whole_loss,
                avg_cls_loss,
                avg_cnt_loss
            )

            eval_loss = self.validation()
            if eval_loss < best_eval_loss:
                best_eval_loss = eval_loss
                best_eval_loss_epoch = epoch
            elif epoch - best_eval_loss_epoch >= self.patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

        training_time = time.time() - start_time
        print(f"Training time: {training_time:.2f} seconds")


class MultitaskLoss(nn.Module):
    def __init__(self, lambda_cnt):
        super().__init__()
        self.lambda_cnt = lambda_cnt
        self.cls_loss_fn = nn.NLLLoss()
        self.cnt_loss_fn = nn.SmoothL1Loss()

    def forward(self, prediciton, target):
        cls_loss = self.cls_loss_fn(prediciton[0], target[0])
        cnt_loss = self.cnt_loss_fn(prediciton[1], target[1])
        return (cls_loss + self.lambda_cnt * cnt_loss, cls_loss, cnt_loss)


class RegressionLoss(nn.Module):
    def __init__(self, lambda_cnt):
        super().__init__()
        self.lambda_cnt = lambda_cnt
        self.cnt_loss_fn = nn.SmoothL1Loss()

    def forward(self, prediciton, target):
        cnt_loss = self.cnt_loss_fn(prediciton[1], target[1])
        return (self.lambda_cnt * cnt_loss, 0.0 * cnt_loss, cnt_loss)


def make_metric_plots(
    trainer: Trainer,
    eval_metrics: list,
    title="Multitask training",
    use_cls_metrics=True,
    use_cnt_metrics=True,
    grid_shape=(2, 2)
):
    cnt_losses = get_metrics_fig({
        "train": trainer.train_cnt_losses,
        "eval": trainer.eval_cnt_losses
    },
        title="Regression Losses"
    )
    cls_losses = get_metrics_fig({
        "train": trainer.train_cls_losses,
        "eval": trainer.eval_cls_losses
    },
        title="Classification Losses"
    )
    figs = [
        metric.plot(metric.values) for metric in eval_metrics
    ]
    if use_cls_metrics:
        figs = [cls_losses] + figs
    if use_cnt_metrics:
        figs = [cnt_losses] + figs

    plot_figs(
        figs,
        grid_shape=grid_shape,
        title=title
    )

### Training

In [ ]:
multitask_loss = MultitaskLoss(1.0)
classification_loss = MultitaskLoss(0.0)
regression_loss = RegressionLoss(1.0)

train_dataloader = DataLoader(
    train_data,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=ImageDataset.collate_fn
)
val_dataloader = DataLoader(
    val_data,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=ImageDataset.collate_fn
)

#### Multitask training

In [ ]:
model_multitask = torch.compile(deepcopy(model).to(device, dtype))
optimizer = optim.Adam(
    model_multitask.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

multitask_eval_metrics = [
    Top1Accuracy(),
    MacroF1Score(),
    PerPairAccuracy(),
    RMSEPerClass(),
    MAEPerClass(),
    RMSE(),
    MAE(),
]

multitask_trainer = Trainer(
    model=model_multitask,
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    loss_fn=multitask_loss,
    eval_metrics=multitask_eval_metrics,
    optimizer=optimizer,
    n_epochs=N_EPOCHS,
    patience=10
)

multitask_trainer.train()

In [ ]:
make_metric_plots(
    multitask_trainer,
    multitask_eval_metrics,
    title="Multitask training",
    use_cls_metrics=True,
    use_cnt_metrics=True,
    grid_shape=(3, 3)
)

#### Classification only training

In [ ]:
model_cls = torch.compile(deepcopy(model).to(device, dtype))
model_cls.forward = model_cls.forward_cls
optimizer = optim.Adam(
    model_cls.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

cls_eval_metrics = [
    Top1Accuracy(),
    MacroF1Score(),
    PerPairAccuracy(),
]

cls_trainer = Trainer(
    model=model_cls,
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    loss_fn=classification_loss,
    eval_metrics=cls_eval_metrics,
    optimizer=optimizer,
    n_epochs=N_EPOCHS,
    patience=10
)

cls_trainer.train()

In [ ]:
make_metric_plots(
    cls_trainer,
    cls_eval_metrics,
    title="Classification only training",
    use_cls_metrics=True,
    use_cnt_metrics=False,
    grid_shape=(2, 2)
)

#### Regression only training

In [ ]:
model_cnt = torch.compile(deepcopy(model).to(device, dtype))
model_cnt.forward = model_cnt.forward_cnt
optimizer = optim.Adam(
    model_cnt.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

cnt_eval_metrics = [
    RMSE(),
    MAE(),
    RMSEPerClass(),
    MAEPerClass(),
]

cnt_trainer = Trainer(
    model=model_cnt,
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    loss_fn=regression_loss,
    eval_metrics=cnt_eval_metrics,
    optimizer=optimizer,
    n_epochs=N_EPOCHS,
    patience=10
)

cnt_trainer.train()

In [ ]:
make_metric_plots(
    cnt_trainer,
    cnt_eval_metrics,
    title="Regression only training",
    use_cls_metrics=False,
    use_cnt_metrics=True,
    grid_shape=(3, 2)
)

### Result analysis

Let's check the confusion matrix for the classification task. I will use the model trained for multitask learning as it had the best accuracy.

In [ ]:
from sklearn.metrics import confusion_matrix

model_multitask.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in val_dataloader:
        x, y_cls, _ = batch
        pred_cls, _ = model_multitask(x)

        all_preds.append(pred_cls)
        all_targets.append(y_cls)

all_preds = torch.argmax(torch.cat(all_preds), dim=-1).cpu().numpy()
all_targets = torch.cat(all_targets).cpu().numpy()

cm = confusion_matrix(all_targets, all_preds, labels=np.arange(N_CLS_CLASSES))

fig = confusion_matrix_fig(
    cm, N_CLS_CLASSES, title="Confusion Matrix - Multitask Model"
)

fig.update_layout(width=PX_SIZE[0], height=PX_SIZE[1])
fig.show()

I will check some missclassified examples. I also want to check how many errors were caused by predicting a good pair but with wrong number of first shape occurences.

In [ ]:
errors = all_preds != all_targets


indexes = np.arange(len(all_targets))
error_indexes = indexes[errors]

n_correct_pairs = 0

for i in error_indexes:
    img, cls, amounts = val_data[i]
    pred_cls = all_preds[i]
    true_pair = class_to_shapes[cls.item()]
    pred_pair = class_to_shapes[pred_cls]
    if true_pair == pred_pair:
        n_correct_pairs += 1

print(f"Number of misclassified samples: {len(error_indexes)}")
print(f"Number of misclassified samples with correct shape pairs: {n_correct_pairs}")


for i in range(3):
    idx = error_indexes[i]
    img, cls, amounts = val_data[idx]
    pred_cls = all_preds[idx]
    true_pair = class_to_shapes[cls.item()]
    pred_pair = class_to_shapes[pred_cls]
    show_image(
        img.to('cpu'),
        title=f"True shape {true_pair}, "
        f"Predicted shapes {pred_pair}",
        figsize=(4, 4)
    )

We can see that for most of the misclassified samples, my model actually predicted a correct pair, but with an incorrect split of elements.